In [6]:
import os
import logging
import numpy as np
import scipy.io as sio
from obspy import UTCDateTime
from obspy.clients.fdsn import Client
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

# Helper: Convert UTC string to MATLAB datenum
def utc_to_datenum(dt_str):
    dt = datetime.strptime(dt_str, "%Y-%m-%d:%H:%M:%S.%f")
    return dt.toordinal() + 366 + (dt.hour + dt.minute / 60 + dt.second / 3600 + dt.microsecond / 3.6e9) / 24

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

def download_trace(net, sta, chan, interval_start, interval_end, client):
    try:
        stream = client.get_waveforms(
            network=net.code,
            station=sta.code,
            location=chan.location_code,
            channel=chan.code,
            starttime=interval_start,
            endtime=interval_end + 1,
            attach_response=True
        )
        stream.trim(starttime=interval_start, endtime=interval_end)
        return (net.code, sta.code, chan.location_code, chan.code, stream)
    except Exception as e:
        logger.warning(f"Download failed for {net.code}.{sta.code}.{chan.location_code}.{chan.code}: {str(e)}")
        return None

def download_seismic_data(
    start_datetime="2022-09-17T05:00:00",
    end_datetime="2022-09-17T10:00:00",
    network="2F",
    station_pattern="AX*",
    location_pattern="*",
    channel_patterns=["HH?", "EL?"],
    output_folder="data",
    username="mczhang8@uw.edu",
    token="RxL42LH6XTNFybHb",
    increment_hours=1,
    overlap_seconds=15
):
    try:
        client = Client("IRIS", user=username, password=token)
        logger.info("Initialized IRIS client")
    except Exception as e:
        logger.error(f"Failed to initialize IRIS client: {str(e)}")
        return {}

    try:
        start = UTCDateTime(start_datetime)
        end = UTCDateTime(end_datetime)
        logger.info(f"Processing range: {start_datetime} to {end_datetime}")
    except Exception as e:
        logger.error(f"Invalid datetime format: {str(e)}")
        return {}

    try:
        inventory = client.get_stations(
            network=network,
            station=station_pattern,
            location=location_pattern,
            channel=",".join(channel_patterns),
            starttime=start,
            endtime=end,
            level="channel"
        )
        logger.info(f"Found {len(inventory.networks)} networks")
    except Exception as e:
        logger.error(f"Failed to retrieve inventory: {str(e)}")
        return {}

    increment_seconds = increment_hours * 3600
    overlap_seconds = float(overlap_seconds)

    last_trace_dict = {}
    current_date = start

    while current_date <= end:
        interval_start = current_date - overlap_seconds
        interval_end = current_date + increment_seconds + overlap_seconds
        date_str = current_date.strftime('%Y-%m-%d-%H-%M-%S')
        logger.info(f"Processing interval: {interval_start} to {interval_end}")

        trace_dict = {
            'network': [], 'station': [], 'location': [], 'channel': [],
            'sensitivity': [], 'sensitivityFrequency': [], 'data': [],
            'sampleCount': [], 'sampleRate': [], 'startTime': [], 'endTime': []
        }

        year = current_date.strftime('%Y')
        month = current_date.strftime('%m')
        output_dir = os.path.join(output_folder, year, month)
        os.makedirs(output_dir, exist_ok=True)

        futures = []
        with ThreadPoolExecutor(max_workers=1) as executor:
            for net in inventory.networks:
                for sta in net.stations:
                    for chan in sta.channels:
                        futures.append(executor.submit(
                            download_trace, net, sta, chan, interval_start, interval_end, client))

            for future in as_completed(futures):
                result = future.result()
                if result is None:
                    continue
                net_code, sta_code, loc_code, chan_code, stream = result

                for trace in stream:
                    try:
                        resp = trace.stats.response._get_overall_sensitivity_and_gain()
                        trace_dict['network'].append(trace.stats.network)
                        trace_dict['station'].append(trace.stats.station)
                        trace_dict['location'].append(trace.stats.location)
                        trace_dict['channel'].append(trace.stats.channel)
                        trace_dict['sensitivity'].append(float(resp[1]))
                        trace_dict['sensitivityFrequency'].append(float(resp[0]))
                        trace_dict['data'].append(trace.data.astype(np.float64))
                        trace_dict['sampleCount'].append(int(trace.stats.npts))
                        trace_dict['sampleRate'].append(float(trace.stats.sampling_rate))
                        trace_dict['startTime'].append(trace.stats.starttime.strftime("%Y-%m-%d:%H:%M:%S.%f"))
                        trace_dict['endTime'].append(trace.stats.endtime.strftime("%Y-%m-%d:%H:%M:%S.%f"))

                        logger.info(
                            f"Processed {trace.stats.network}.{trace.stats.station}.{trace.stats.channel}: "
                            f"{trace.stats.npts} samples, {trace.stats.sampling_rate} Hz, "
                            f"from {trace.stats.starttime} to {trace.stats.endtime}"
                        )
                    except Exception as e:
                        logger.error(f"Error processing trace {trace.id}: {str(e)}")

        if trace_dict['network']:
            mat_file = os.path.join(output_dir, f"{date_str}_fast.mat")
            try:
                num_traces = len(trace_dict['network'])
                dtype = [
                    ('network', 'O'), ('station', 'O'), ('location', 'O'), ('channel', 'O'),
                    ('sensitivity', 'f8'), ('sensitivityFrequency', 'f8'), ('data', 'O'),
                    ('sampleCount', 'i4'), ('sampleRate', 'f8'),
                    ('startTime', 'f8'), ('endTime', 'f8')
                ]

                trace_struct = np.zeros(num_traces, dtype=dtype)
                for i in range(num_traces):
                    trace_struct[i] = (
                        trace_dict['network'][i],
                        trace_dict['station'][i],
                        trace_dict['location'][i],
                        trace_dict['channel'][i],
                        trace_dict['sensitivity'][i],
                        trace_dict['sensitivityFrequency'][i],
                        trace_dict['data'][i],
                        trace_dict['sampleCount'][i],
                        trace_dict['sampleRate'][i],
                        utc_to_datenum(trace_dict['startTime'][i]),
                        utc_to_datenum(trace_dict['endTime'][i])
                    )

                sio.savemat(mat_file, {'trace': trace_struct}, format='5', do_compression=True)
                file_size = os.path.getsize(mat_file) / 1024
                logger.info(f"Saved {mat_file} ({file_size:.2f} KB), {num_traces} traces")
                last_trace_dict = trace_dict
            except Exception as e:
                logger.error(f"Failed to save {mat_file}: {str(e)}")

        current_date += increment_seconds

    return last_trace_dict

if __name__ == "__main__":
    trace_data = download_seismic_data()
    if trace_data.get('network'):
        logger.info("Data download complete.")
    else:
        logger.warning("No data downloaded.")


2025-05-21 10:19:44,526 - INFO - Initialized IRIS client
2025-05-21 10:19:44,528 - INFO - Processing range: 2022-09-17T05:00:00 to 2022-09-17T10:00:00
2025-05-21 10:19:44,776 - INFO - Found 1 networks
2025-05-21 10:19:44,777 - INFO - Processing interval: 2022-09-17T04:59:45.000000Z to 2022-09-17T06:00:15.000000Z
2025-05-21 10:19:49,088 - INFO - Processed 2F.AX01A.HH1: 726001 samples, 200.0 Hz, from 2022-09-17T04:59:45.000200Z to 2022-09-17T06:00:15.000200Z
2025-05-21 10:19:49,092 - INFO - Processed 2F.AX01A.HH2: 726001 samples, 200.0 Hz, from 2022-09-17T04:59:45.000200Z to 2022-09-17T06:00:15.000200Z
2025-05-21 10:19:49,101 - INFO - Processed 2F.AX01A.HHZ: 726001 samples, 200.0 Hz, from 2022-09-17T04:59:45.000200Z to 2022-09-17T06:00:15.000200Z
2025-05-21 10:19:49,108 - INFO - Processed 2F.AX02A.EL1: 726000 samples, 200.0 Hz, from 2022-09-17T04:59:45.004498Z to 2022-09-17T06:00:14.999498Z
2025-05-21 10:19:49,585 - INFO - Processed 2F.AX02A.EL2: 726000 samples, 200.0 Hz, from 2022-09-17